In [ ]:
import pandas as pd
from IPython.display import display, clear_output
import ipywidgets as widgets
import json

data_loc = "../data/synthetic/components"

### Set up dropdowns

In [ ]:
with open(f'{data_loc}/motherboards.json', 'r') as file:
    mobo_data = json.load(file)

mobo_choices = [(f"{mobo['brand']} {mobo['model']}", mobo['id']) for mobo in mobo_data]

In [ ]:
# everyone but mobo has id + name
# function time baby

def get_component_choices(component):
    with open(f'{data_loc}/{component}.json', 'r') as file:
        component_data = json.load(file)
    return [(f"{item['name']}", item['id']) for item in component_data]

In [ ]:
# also all components need the same stuff for the dropdowns, so let's make a function for that too

def create_dropdown(component, choices=None, display_it=False):
    choices = choices if choices is not None else get_component_choices(component)
    dropdown = widgets.Select(
        options=choices,
        value=choices[0][1],
        description=f'{component}:',
        disabled=False,
        layout=widgets.Layout(width='max-content'),
        rows=len(choices),
    )
    if display_it == True: display(dropdown)
    return dropdown

# note to future self: mobo_dropdown.value to get selected ID, mobo_dropdown.label to get selected label

In [ ]:
mobo_dropdown = create_dropdown('motherboards', mobo_choices)

component_list = ["cases", "coolers", "cpus", "fans", "gpus", "lighting", "psus", "ram", "storage"]
dropdowns = [create_dropdown(component) for component in component_list]
dropdowns.insert(0, mobo_dropdown)

I want to check whether components are compatible. There are a couple of ways to do this. I could poll with every selection, or I can just provide a big fat button so you submit the build, then check. 

Performance wise, the latter wins, but I don't think any of this is performance intensive enough to warant a hack. 

Ok, gonna go for polling. 

I also think that gives more opportunity for learning.

### Set up observers for dropdown choices

In [ ]:
# add an observer for every dropdown
def update_build_text(change):
    build_text.value = f"{newline.join([dropdown.value for dropdown in dropdowns])}"

for dropdown in dropdowns:
    dropdown.observe(update_build_text, names="value")

I'll also need to actually check whether everything is actually compatible.

### Compatibility checking

In [ ]:
# CASE COMPATIBILITY
def is_case_compatible():
    # check that mobo form factor is supported
    # check that gpu length is supported
    # check that cpu height is supported
    # check that psu form factor is supported
    # check that radiator size is supported
    # check that sata 2.5 is supported
    # check that sata 3.5 is supported
    return True

In [ ]:
# loads our entire catalog from the data folder
CATEGORIES = ['cases', 'coolers', 'cpus', 'fans', 'gpus', 'lighting', 'motherboards', 'psus', 'ram', 'storage']
CATALOG = {}
for category in CATEGORIES:
    with open(f'{data_loc}/{category}.json', 'r') as file:
        cat_data = json.load(file)
    CATALOG[category] = cat_data

# I'll need a way to grab the current build
def get_current_build():
    return { dropdown.description[:-1]: dropdown.value for dropdown in dropdowns }


def check_build():
    return is_case_compatible()

### Display

In [ ]:
newline = "\n"
build_text = widgets.Textarea(
    value=f"{newline.join([dropdown.value for dropdown in dropdowns])}",
    placeholder='Type something',
    description='Build:',
    disabled=True,
    rows=len(dropdowns),
    layout=widgets.Layout(justify_content='flex-end', width='max-content')
)

In [ ]:
widgets.GridBox(
    children=dropdowns + [build_text],
    layout=widgets.Layout(
        # width='50%',
        grid_template_columns='auto auto auto',
        grid_template_rows='auto auto auto auto', 
        grid_gap='5px 10px',
    )
)